# Eucrisa Copay Fraud Signals — 02 · Score lookalikes

**Purpose.** The rule engine already tells us which pharmacies tripped a signal — those are
handled by the rule-based workflow. This notebook adds the piece the rules *can't* give us:
the **lookalikes** — pharmacies that tripped **nothing** but whose behavioral profile
resembles flagged pharmacies (high model probability).

Flow:
1. Run the rule engine → separate **rule-flagged** (handoff to rule workflow) from the rest.
2. Load the trained models and score the full population.
3. Keep `lookalike_flag` pharmacies (high `signal_probability`, tripped 0 rules).
4. Apply the **closest-signal** and **recency** layers to the lookalikes **only**, to triage them.

> Run **01_train_models.ipynb** first so `models/eucrisa_models.joblib` exists.

## 1. Setup + load data

In [1]:
from nurtec_pipeline import *
import pandas as pd, numpy as np

df = load_copay(COPAY_PATH)
df = attach_pharmacy_type(df)
ref_month = df['month'].max()
ris = load_risrx(RISRX_PATH, data_min=df['fill_date'].min(), data_max=df['fill_date'].max())
print(f"{len(df):,} claims | {df['pharmacy_key'].nunique():,} pharmacies | snapshot {pd.Timestamp(ref_month).date()}")

964,206 claims | 31,877 pharmacies | snapshot 2026-07-01


In [2]:
import importlib
import nurtec_pipeline
importlib.reload(nurtec_pipeline)
from nurtec_pipeline import *   # re-pull names into your namespace
print(COPAY_PATH)

COPAY_EUCRISA_CLEAN.csv


## 2. Rule engine (the rule-based output)
`rule_flagged` is what the rule workflow already owns — exported here purely for the handoff.
Our ML focus is everything *not* in this set.

In [3]:
score, detail = run_rule_engine(df, ris)
s1, s2, s3, s4_monthly, s7 = detail['s1'], detail['s2'], detail['s3'], detail['s4_monthly'], detail['s7']

rule_flagged = (score[score['signals_triggered'] > 0]
                .sort_values('signals_triggered', ascending=False))
print(f"rule-flagged pharmacies: {len(rule_flagged):,}")

Signal 1 (CPU vs chain baseline) ...
  flagged: 3709
Signal 2 (cost per claim at max) ...
  flagged: 40
Signal 3 (CPU pre/post RisRx) ...
  flagged rebounds: 1
  flagged pharmacies: 3555
Signal 7 (popup/dormant) ...
  flagged pharmacies: 330
Signal 5 (high HCP utilization at 3,838 flagged pharmacies) ...
  flagged pharmacies: 1245
Building pharmacy features ...

Rule engine: 3,838 pharmacies tripped >=1 signal; 1,288 tripped >=2.
    signal1_cpu_above_chain    3,709
    signal2_cost_at_max        40
    signal3_cpu_rebound        1
    signal5_high_hcp           1,245
    signal7_popup              330
rule-flagged pharmacies: 3,838


## 3. Load models + score every pharmacy
Anomaly score and classifier probability are added with the **persisted** models — no
refitting, so results are reproducible against the trained snapshot.

In [4]:
bundle = load_model_bundle()
score = apply_anomaly_model(score, bundle['anomaly'])
score = apply_signal_classifier(score, bundle['classifier'])   # -> signal_probability, lookalike_flag
print(f"lookalikes (high prob, tripped 0 rules): {int(score['lookalike_flag'].sum()):,} "
      f"(threshold {bundle['metadata']['lookalike_prob_threshold']})")

Loaded model bundle trained at 2026-07-17T15:06:39 on 31877 pharmacies (ref month 2026-07-01).
lookalikes (high prob, tripped 0 rules): 2,192 (threshold 0.5)


## 4. Isolate the lookalikes
From here on we work **only** on the lookalike subset — the rule-flagged pharmacies are
already covered by the rule workflow.

In [5]:
look = score[score['lookalike_flag']].copy()
print(f"{len(look):,} lookalikes to triage")

2,192 lookalikes to triage


## 5. Closest-signal layer (lookalikes only)
Two complementary views of *which* signal each lookalike leans toward:
`prox_signal*` (distance to the real threshold) and `p_signal*` (learned propensity).

In [6]:
look = signal_proximity(look, s3, s7.attrs['high_vol_threshold'])
look = apply_per_signal_propensity(look, bundle['propensity'])
print(look['closest_signal'].value_counts().to_string())

closest_signal
Signal 7 (Popup)           2059
Signal 2 (Cost at max)      124
Signal 1 (CPU vs chain)       8
Signal 3 (CPU rebound)        1


## 6. Recency layer (lookalikes only)
For lookalikes there is no *triggered* date, so we date each one by the most recent
**near-miss** month of its closest signal — letting the queue surface freshly-suspicious
pharmacies first.

In [7]:
rec = compute_signal_recency(df, s1, s2, s4_monthly)
nm_cols = ['pharmacy_key','last_fill_month','s1_nearmiss_month','s2_nearmiss_month','s4_nearmiss_month']
look = look.drop(columns=[c for c in nm_cols if c != 'pharmacy_key' and c in look], errors='ignore')
look = look.merge(rec[[c for c in nm_cols if c in rec]], on='pharmacy_key', how='left')
look = lookalike_recency(look, ref_month)
look['months_since_closest_signal'].describe()

count    2192.000000
mean       53.077555
std        29.017872
min         0.000000
25%        27.000000
50%        62.000000
75%        79.000000
max        90.000000
Name: months_since_closest_signal, dtype: float64

## 7. Triage queue + export
Sorted: most recent near-miss first, then closest to a threshold, then highest model
probability. Written alongside the rule-flagged handoff sheet.

In [8]:
triage_cols = ['pharmacy_key','total_claims','claims_per_month','anomaly_score',
               'signal_probability','months_since_last_fill','closest_signal',
               'closest_signal_event_month','months_since_closest_signal',
               'closest_proximity','closest_reason',
               'prox_signal1','prox_signal2','prox_signal3','prox_signal4','prox_signal5','prox_signal7',
               'p_signal1','p_signal2','p_signal3','p_signal4','p_signal5','p_signal7']
triage_cols = [c for c in triage_cols if c in look.columns]

triage = look.sort_values(['months_since_closest_signal','closest_proximity','signal_probability'],
                          ascending=[True, False, False])[triage_cols]

print(f"within 80% of a trigger: {(look['closest_proximity'] >= 0.8).sum():,}")
print(f"within 95% of a trigger: {(look['closest_proximity'] >= 0.95).sum():,}")

with pd.ExcelWriter(LOOKALIKE_XLSX, engine='openpyxl') as xw:
    triage.to_excel(xw, sheet_name='Lookalike_Triage', index=False)
    rule_flagged.to_excel(xw, sheet_name='Rule_Flagged_Handoff', index=False)
print(f'Wrote {LOOKALIKE_XLSX}')
triage.head(20)

within 80% of a trigger: 44
within 95% of a trigger: 3
Wrote Eucrisa_Lookalike_Triage.xlsx


,pharmacy_key,total_claims,claims_per_month,anomaly_score,signal_probability,closest_signal,closest_signal_event_month,months_since_closest_signal,closest_proximity,closest_reason,...,prox_signal3,prox_signal4,prox_signal5,prox_signal7,p_signal1,p_signal2,p_signal3,p_signal4,p_signal5,p_signal7
1478,WALGREENS 21202,5,1.250000,0.282318,0.699653,Signal 2 (Cost at max),2026-07-01,0,0.500000,40% of claims at/near program max (fires at 80%),...,NaN,0.0,NaN,0.2,0.734365,NaN,NaN,NaN,0.009711,0.000000
1496,WALGREENS DRUG STORE 00283,23,1.769231,0.477009,0.628508,Signal 2 (Cost at max),2026-07-01,0,0.489130,39% of claims at/near program max (fires at 80%),...,NaN,0.0,NaN,0.2,0.595802,NaN,NaN,NaN,0.480306,0.012782
18,AMAZON PHARMACY 005,9,4.500000,0.605421,0.793588,Signal 2 (Cost at max),2026-06-01,1,0.833333,67% of claims at/near program max (fires at 80%),...,NaN,0.0,NaN,0.5,0.719647,NaN,NaN,NaN,0.568824,0.186104
1380,WAL MART SUPERCENTER 1720,3,1.500000,0.430955,0.640209,Signal 2 (Cost at max),2026-06-01,1,0.833333,67% of claims at/near program max (fires at 80%),...,NaN,0.0,NaN,0.2,0.585791,NaN,NaN,NaN,0.000000,0.000000
2111,WALMART 826,9,3.000000,0.407220,0.734279,Signal 7 (Popup),2026-06-01,1,0.700000,best gap 1.0 mo then 1 claims; new-launch wind...,...,NaN,0.0,NaN,0.7,0.744912,NaN,NaN,NaN,0.243036,0.009617
1467,WALGREENS 15116,16,2.285714,0.453068,0.693244,Signal 7 (Popup),2026-06-01,1,0.700000,best gap 5.0 mo then 7 claims; new-launch wind...,...,NaN,0.0,NaN,0.7,0.677611,NaN,NaN,NaN,0.453779,0.006544
181,CVS PHARMACY 01119,23,2.090909,0.417126,0.644626,Signal 7 (Popup),2026-06-01,1,0.700000,best gap 11.0 mo then 7 claims; new-launch win...,...,NaN,0.0,NaN,0.7,0.645765,NaN,NaN,NaN,0.453683,0.034819
10,ACME PHARMACY 1856,3,1.000000,0.277085,0.766384,Signal 2 (Cost at max),2026-06-01,1,0.416667,33% of claims at/near program max (fires at 80%),...,NaN,0.0,NaN,0.1,0.774309,NaN,NaN,NaN,0.000000,0.000000
890,PUBLIX PHARMACY 0682,4,4.000000,0.535201,0.673315,Signal 7 (Popup),2026-06-01,1,0.400000,best gap 0.0 mo then 0 claims; new-launch wind...,...,NaN,0.0,NaN,0.4,0.659391,NaN,NaN,NaN,0.008255,0.003136
1809,WALGREENS DRUG STORE 09430,11,1.222222,0.226688,0.583632,Signal 2 (Cost at max),2026-06-01,1,0.340909,27% of claims at/near program max (fires at 80%),...,NaN,0.0,NaN,0.2,0.634564,NaN,NaN,NaN,0.192962,0.000000


In [11]:
import nurtec_pipeline as P
import pandas as pd, numpy as np

# ---- 1. Full-population scoring with recency (run the full chain) ----
score, detail = P.run_rule_engine(df, ris)
bundle = P.load_model_bundle()
score = P.apply_anomaly_model(score, bundle["anomaly"])
score = P.apply_signal_classifier(score, bundle["classifier"])
score = P.apply_per_signal_propensity(score, bundle["propensity"])
score = P.signal_proximity(score, detail["s3"], detail["s7"].attrs["high_vol_threshold"])

# recency: attach event months -> months_since_signal + last_signal_month
rec = P.compute_signal_recency(df, detail["s1"], detail["s2"], detail["s4_monthly"])
score = score.merge(rec, on="pharmacy_key", how="left")

# apply_recency expects signal4 columns; run_rule_engine doesn't merge them -> add safe defaults
if "signal4_qty_spike" not in score.columns:
    score["signal4_qty_spike"] = False
if "signal4_event_month" not in score.columns:
    score["signal4_event_month"] = pd.NaT

score = P.apply_recency(score, detail["ref_month"])   # -> last_signal_month, months_since_signal

# day-level last signal date for precise 1-week/1-month windows
lsm_map = score.set_index("pharmacy_key")["last_signal_month"]
lsd = P.compute_last_signal_date(df, lsm_map).rename("last_signal_date")
score = score.merge(lsd, on="pharmacy_key", how="left")

AS_OF = df["fill_date"].max()   # anchor = latest fill in the data
score["detected_days_ago"] = (AS_OF - pd.to_datetime(score["last_signal_date"])).dt.days

# ---- 2. Flagged pharmacies only ----
flagged = score[score["signals_triggered"] > 0].copy()

# ---- 3. Funnel windows ----
windows = {"last_1_year": 365, "last_1_month": 30, "last_1_week": 7}

detail_cols = [
    "pharmacy_key", "signals_triggered",
    "signal1_cpu_above_chain", "signal2_cost_at_max", "signal3_cpu_rebound",
    "signal5_high_hcp", "signal7_popup",
    "total_claims", "total_benefit", "avg_cpu", "avg_cpu_pct_wac",
    "pct_claims_at_max", "cpu_dev_vs_chain", "n_high_hcps",
    "reactivation_window_vol", "dormancy_months", "new_window_vol",
    "signal_probability", "anomaly_score",
    "closest_signal", "closest_proximity", "closest_reason",
    "last_signal_month", "last_signal_date", "detected_days_ago",
    "months_since_signal", "on_risrx",
]
detail_cols = [c for c in detail_cols if c in flagged.columns]

print("="*60)
print("FUNNEL — flagged pharmacies detected within each window")
print("="*60)
print(f"Anchor (today) = {AS_OF.date()}")
print(f"All-time flagged pharmacies: {len(flagged):,}\n")

with pd.ExcelWriter("Eucrisa_Funnel_Detected.xlsx", engine="openpyxl") as xw:
    # all-time sheet for reference
    flagged[detail_cols].sort_values("detected_days_ago").to_excel(
        xw, "all_time_flagged", index=False)
    for name, days in windows.items():
        sub = flagged[flagged["detected_days_ago"] <= days].copy()
        sub = sub.sort_values(["detected_days_ago", "signals_triggered"],
                              ascending=[True, False])
        print(f"{name:<14}: {len(sub):,} pharmacies | "
              f"{int(sub['total_claims'].sum()):,} claims | "
              f"${sub['total_benefit'].sum():,.0f} benefit")
        # per-signal counts in this window
        for f in ["signal1_cpu_above_chain","signal2_cost_at_max","signal3_cpu_rebound",
                  "signal5_high_hcp","signal7_popup"]:
            if f in sub: print(f"      {f:<26} {int(sub[f].sum()):,}")
        sub[detail_cols].to_excel(xw, name, index=False)
        print()

print("Wrote Eucrisa_Funnel_Detected.xlsx")

Signal 1 (CPU vs chain baseline) ...
  flagged: 3709
Signal 2 (cost per claim at max) ...
  flagged: 40
Signal 3 (CPU pre/post RisRx) ...
  flagged rebounds: 1
  flagged pharmacies: 3555
Signal 7 (popup/dormant) ...
  flagged pharmacies: 330
Signal 5 (high HCP utilization at 3,838 flagged pharmacies) ...
  flagged pharmacies: 1245
Building pharmacy features ...

Rule engine: 3,838 pharmacies tripped >=1 signal; 1,288 tripped >=2.
    signal1_cpu_above_chain    3,709
    signal2_cost_at_max        40
    signal3_cpu_rebound        1
    signal5_high_hcp           1,245
    signal7_popup              330
Loaded model bundle trained at 2026-07-17T15:06:39 on 31877 pharmacies (ref month 2026-07-01).
FUNNEL — flagged pharmacies detected within each window
Anchor (today) = 2026-07-05
All-time flagged pharmacies: 3,838

last_1_year   : 1,044 pharmacies | 334,917 claims | $66,622,164 benefit
      signal1_cpu_above_chain    997
      signal2_cost_at_max        16
      signal3_cpu_rebound     

In [14]:
import nurtec_pipeline as P
import pandas as pd, numpy as np

# ---- 1. isolate ML lookalikes (high prob, tripped 0 rules) ----
look = score[score["lookalike_flag"]].copy()
print(f"Total ML lookalikes: {len(look):,}")

# ---- 2. date each lookalike by its closest-signal near-miss month ----
# compute_signal_recency already merged (s1/s2/s4 near-miss + last_fill_month) into score;
# pull the near-miss cols + s7 episode + s5 event that lookalike_recency needs.
nm_cols = ["pharmacy_key","last_fill_month","s1_nearmiss_month",
           "s2_nearmiss_month","s4_nearmiss_month",
           "s7_best_episode_month","signal5_event_month"]
have = [c for c in nm_cols if c in score.columns]
look = look.drop(columns=[c for c in have if c != "pharmacy_key" and c in look.columns],
                 errors="ignore")
look = look.merge(score[have], on="pharmacy_key", how="left")

look = P.lookalike_recency(look, detail["ref_month"])   # -> months_since_closest_signal

# convert months -> approx days for the window filter
look["detected_days_ago_ml"] = look["months_since_closest_signal"] * 30.44

# ---- 3. verification columns ----
ml_cols = [
    "pharmacy_key", "total_claims", "total_benefit",
    "signal_probability", "anomaly_score",
    "closest_signal", "closest_proximity", "closest_reason",
    "closest_signal_event_month", "months_since_closest_signal",
    "detected_days_ago_ml", "on_risrx",
    "p_signal1","p_signal2","p_signal3","p_signal5","p_signal7",
]
ml_cols = [c for c in ml_cols if c in look.columns]

# ---- 4. funnel ----
print("="*60)
print("ML LOOKALIKE FUNNEL — detected within each window")
print("="*60)
print(f"Anchor (today) = {AS_OF.date()}")
print(f"All-time ML lookalikes: {len(look):,}\n")

with pd.ExcelWriter("Eucrisa_Funnel_ML_Lookalikes.xlsx", engine="openpyxl") as xw:
    look[ml_cols].sort_values("detected_days_ago_ml").to_excel(
        xw, "all_time_lookalikes", index=False)
    for name, days in windows.items():
        sub = look[look["detected_days_ago_ml"] <= days].copy()
        sub = sub.sort_values(["detected_days_ago_ml","signal_probability"],
                              ascending=[True, False])
        print(f"{name:<14}: {len(sub):,} lookalikes | "
              f"{int(sub['total_claims'].sum()):,} claims | "
              f"${sub['total_benefit'].sum():,.0f} benefit")
        # closest-signal breakdown within window
        if len(sub):
            for sig, cnt in sub["closest_signal"].value_counts().items():
                print(f"      {sig:<26} {cnt:,}")
        sub[ml_cols].to_excel(xw, name, index=False)
        print()

print("Wrote Eucrisa_Funnel_ML_Lookalikes.xlsx")

Total ML lookalikes: 2,192
ML LOOKALIKE FUNNEL — detected within each window
Anchor (today) = 2026-07-05
All-time ML lookalikes: 2,192

last_1_year   : 313 lookalikes | 12,477 claims | $2,270,943 benefit
      Signal 7 (Popup)           271
      Signal 2 (Cost at max)     39
      Signal 1 (CPU vs chain)    2
      Signal 3 (CPU rebound)     1

last_1_month  : 2 lookalikes | 28 claims | $16,997 benefit
      Signal 2 (Cost at max)     2

last_1_week   : 2 lookalikes | 28 claims | $16,997 benefit
      Signal 2 (Cost at max)     2

Wrote Eucrisa_Funnel_ML_Lookalikes.xlsx


In [16]:
import pandas as pd

# ---------- reuses from earlier cells: df, score, flagged, look, windows, AS_OF ----------

# full population totals
total_pharm   = len(score)
red           = df[(~df["is_reversal"]) & (df["QTY_DISPENSED"] > 0)]
total_claims  = len(red)
total_benefit = red["BENEFIT_PAID"].sum()
n_flagged     = len(flagged)          # rule-based
n_lookalike   = len(look)             # ML lookalikes
n_pred1       = n_flagged + n_lookalike  # predicted class 1 (rules + lookalikes)

def win_counts(frame, col):
    return {w: frame[frame[col] <= d] for w, d in windows.items()}

rule_w = win_counts(flagged, "detected_days_ago")
ml_w   = win_counts(look,    "detected_days_ago_ml")

L = []
L.append("EUCRISA LOOKALIKE MODEL — DETECTION SUMMARY")
L.append(f"Data: {df['fill_date'].min().date()} -> {df['fill_date'].max().date()}  |  anchor {AS_OF.date()}")
L.append("")
L.append("FULL POPULATION")
L.append(f"  Pharmacies       : {total_pharm:,}")
L.append(f"  Claims           : {total_claims:,}")
L.append(f"  Benefit          : ${total_benefit:,.0f}")
L.append(f"  Predicted class 1: {n_pred1:,}")
L.append(f"    - Rule-flagged : {n_flagged:,}")
L.append(f"    - ML lookalikes: {n_lookalike:,}")
L.append("")
L.append("DETECTION FUNNEL (pharmacies detected within window)")
L.append(f"  {'Window':<12}{'Rule-flagged':>14}{'ML lookalikes':>15}{'Total':>10}")
L.append("  " + "-"*50)
L.append(f"  {'all_time':<12}{n_flagged:>14,}{n_lookalike:>15,}{n_pred1:>10,}")
for w in windows:
    r, m = len(rule_w[w]), len(ml_w[w])
    L.append(f"  {w:<12}{r:>14,}{m:>15,}{r+m:>10,}")
L.append("")
L.append("ML LOOKALIKES — claims & benefit by window")
L.append(f"  {'Window':<12}{'Lookalikes':>12}{'Claims':>12}{'Benefit':>16}")
L.append("  " + "-"*52)
L.append(f"  {'all_time':<12}{n_lookalike:>12,}{int(look['total_claims'].sum()):>12,}"
         f"{'$'+format(int(look['total_benefit'].sum()),','):>16}")
for w in windows:
    s = ml_w[w]
    L.append(f"  {w:<12}{len(s):>12,}{int(s['total_claims'].sum()):>12,}"
             f"{'$'+format(int(s['total_benefit'].sum()),','):>16}")

summary = "\n".join(L)

# print with Teams code-fence so columns stay aligned when pasted
print("```")
print(summary)
print("```")

with open("Eucrisa_Teams_Summary.txt", "w") as fh:
    fh.write("```\n" + summary + "\n```\n")
print("\nSaved Eucrisa_Teams_Summary.txt")

```
EUCRISA LOOKALIKE MODEL — DETECTION SUMMARY
Data: 2018-05-31 -> 2026-07-05  |  anchor 2026-07-05

FULL POPULATION
  Pharmacies       : 31,877
  Claims           : 964,206
  Benefit          : $182,212,221
  Predicted class 1: 6,030
    - Rule-flagged : 3,838
    - ML lookalikes: 2,192

DETECTION FUNNEL (pharmacies detected within window)
  Window        Rule-flagged  ML lookalikes     Total
  --------------------------------------------------
  all_time             3,838          2,192     6,030
  last_1_year          1,044            313     1,357
  last_1_month           369              2       371
  last_1_week            167              2       169

ML LOOKALIKES — claims & benefit by window
  Window        Lookalikes      Claims         Benefit
  ----------------------------------------------------
  all_time           2,192      98,728     $15,078,047
  last_1_year          313      12,477      $2,270,942
  last_1_month           2          28         $16,996
  last_1_week 

In [15]:
import pandas as pd

# reuses: df, score, flagged, windows, AS_OF from the funnel cell above
total_pharm   = len(score)
total_claims  = len(df[(~df["is_reversal"]) & (df["QTY_DISPENSED"] > 0)])
total_benefit = df[(~df["is_reversal"]) & (df["QTY_DISPENSED"] > 0)]["BENEFIT_PAID"].sum()
n_flagged     = len(flagged)
n_lookalike   = int(score["lookalike_flag"].sum()) if "lookalike_flag" in score else 0

sig_flags = ["signal1_cpu_above_chain","signal2_cost_at_max","signal3_cpu_rebound",
             "signal5_high_hcp","signal7_popup"]

lines = []
lines.append("="*52)
lines.append("EUCRISA — MODEL DETECTION SUMMARY")
lines.append("="*52)
lines.append(f"Data range        : {df['fill_date'].min().date()} -> {df['fill_date'].max().date()}")
lines.append(f"Anchor (today)    : {AS_OF.date()}")
lines.append("")
lines.append("FULL POPULATION")
lines.append(f"  Total pharmacies : {total_pharm:,}")
lines.append(f"  Total claims     : {total_claims:,}")
lines.append(f"  Total benefit    : ${total_benefit:,.0f}")
lines.append(f"  Flagged (>=1 sig): {n_flagged:,} ({n_flagged/total_pharm:.1%})")
lines.append(f"  Lookalikes       : {n_lookalike:,}")
lines.append("")
lines.append("FUNNEL — flagged pharmacies detected within window")
lines.append(f"  {'Window':<14}{'Pharmacies':>12}{'Claims':>12}{'Benefit':>16}")
lines.append("  " + "-"*52)
# all-time row
lines.append(f"  {'all_time':<14}{n_flagged:>12,}"
             f"{int(flagged['total_claims'].sum()):>12,}"
             f"{'$'+format(int(flagged['total_benefit'].sum()),','):>16}")
for name, days in windows.items():
    sub = flagged[flagged["detected_days_ago"] <= days]
    lines.append(f"  {name:<14}{len(sub):>12,}"
                 f"{int(sub['total_claims'].sum()):>12,}"
                 f"{'$'+format(int(sub['total_benefit'].sum()),','):>16}")
lines.append("")
lines.append("PER-SIGNAL BREAKDOWN BY WINDOW (pharmacies)")
hdr = f"  {'Signal':<26}{'all_time':>10}" + "".join(f"{w.replace('last_','').replace('_',''):>10}" for w in windows)
lines.append(hdr)
lines.append("  " + "-"*66)
subsets = {"all_time": flagged}
subsets.update({w: flagged[flagged["detected_days_ago"] <= d] for w, d in windows.items()})
for f in sig_flags:
    row = f"  {f.replace('signal','S').replace('_',' '):<26}"
    row += f"{int(flagged[f].sum()):>10}"
    for w in windows:
        row += f"{int(subsets[w][f].sum()):>10}"
    lines.append(row)
lines.append("="*52)

summary = "\n".join(lines)
print(summary)

# also save to a .txt so you can copy-paste into Teams
with open("Eucrisa_Funnel_Summary.txt", "w") as fh:
    fh.write(summary)
print("\nSaved Eucrisa_Funnel_Summary.txt")

EUCRISA — MODEL DETECTION SUMMARY
Data range        : 2018-05-31 -> 2026-07-05
Anchor (today)    : 2026-07-05

FULL POPULATION
  Total pharmacies : 31,877
  Total claims     : 964,206
  Total benefit    : $182,212,221
  Flagged (>=1 sig): 3,838 (12.0%)
  Lookalikes       : 2,192

FUNNEL — flagged pharmacies detected within window
  Window          Pharmacies      Claims         Benefit
  ----------------------------------------------------
  all_time             3,838     677,120    $150,168,459
  last_1_year          1,044     334,917     $66,622,164
  last_1_month           369     284,829     $52,239,514
  last_1_week            167     239,325     $41,040,158

PER-SIGNAL BREAKDOWN BY WINDOW (pharmacies)
  Signal                      all_time     1year    1month     1week
  ------------------------------------------------------------------
  S1 cpu above chain              3709       997       336       141
  S2 cost at max                    40        16         7         3
  S3 cp

In [1]:
import nurtec_pipeline as P
import pandas as pd

# ---------- 1. LOAD ----------
df  = P.load_copay("COPAY_EUCRISA_CLEAN.csv")
ris = P.load_risrx("RisRx_List.csv",
                   data_min=df["fill_date"].min(),
                   data_max=df["fill_date"].max())

# ---------- 2. RULE ENGINE ----------
score, detail = P.run_rule_engine(df, ris)
high_vol_threshold = detail["s7"].attrs["high_vol_threshold"]

# ---------- 3. ML: classifier + proximity ----------
data, metrics, importances, clf = P.train_signal_classifier(score)
out = P.signal_proximity(data, detail["s3"], high_vol_threshold)

# =====================================================================
#                       PPT NUMBERS
# =====================================================================
red = df[(~df["is_reversal"]) & (df["QTY_DISPENSED"] > 0)]
pred1 = out[out["signal_probability"] >= 0.50]

print("="*55)
print("SLIDE 1 — MODEL OVERVIEW")
print("="*55)
print(f"Drug                 : Eucrisa")
print(f"Data range           : {df['fill_date'].min().date()} → {df['fill_date'].max().date()}")
print(f"Total claims (raw)    : {len(df):,}")
print(f"Total claims (modeled): {len(red):,}")
print(f"Pharmacies (modeled)  : {len(score):,}")
print(f"Model features        : {len(P.CLF_FEATURES)}")
print(f"Predicted class 1     : {len(pred1):,}")
print(f"\n12 features: {P.CLF_FEATURES}")

print("\n" + "="*55)
print("SLIDE 2 — METHODOLOGY & VALIDATION")
print("="*55)
print(f"Label positives (any signal): {metrics['n_positive']:,} "
      f"({metrics['positive_rate']:.1%})")
print(f"ROC-AUC (test)              : {metrics['roc_auc']:.3f}")
print(f"PR-AUC  (test)             : {metrics['pr_auc']:.3f}")

# --- signals used in the label ---
print(f"\nSignals in label: {detail['signal_flags']}")
for f in detail["signal_flags"]:
    print(f"    {f:<26} flagged: {int(score[f].sum()):,}")

# --- TP vs lookalikes among predicted class 1 ---
tp        = int((pred1["signals_triggered"] > 0).sum())
lookalike = int((pred1["signals_triggered"] == 0).sum())
print(f"\nPredicted class 1 : {len(pred1):,}")
print(f"  True positives  : {tp:,}  (triggered >=1 rule)")
print(f"  Lookalikes      : {lookalike:,}  (no rule, high score)")

# --- RisRx overlap among predicted class 1 ---
on_ris = int(pred1["on_risrx"].sum())
print(f"\nRisRx overlap in predicted class 1: {on_ris}")
print(f"  (model 'caught' all of them by definition: {on_ris}/{on_ris})")

# --- lookalike closest-signal breakdown ---
look = pred1[pred1["signals_triggered"] == 0]
print(f"\nLookalikes by closest signal:")
print(look["closest_signal"].value_counts().to_string())

Signal 1 (CPU vs chain baseline) ...
  flagged: 3709
Signal 2 (cost per claim at max) ...
  flagged: 40
Signal 3 (CPU pre/post RisRx) ...
  flagged rebounds: 1
  flagged pharmacies: 3555
Signal 7 (popup/dormant) ...
  flagged pharmacies: 330
Signal 5 (high HCP utilization at 3,838 flagged pharmacies) ...
  flagged pharmacies: 1245
Building pharmacy features ...

Rule engine: 3,838 pharmacies tripped >=1 signal; 1,288 tripped >=2.
    signal1_cpu_above_chain    3,709
    signal2_cost_at_max        40
    signal3_cpu_rebound        1
    signal5_high_hcp           1,245
    signal7_popup              330
SLIDE 1 — MODEL OVERVIEW
Drug                 : Eucrisa
Data range           : 2018-05-31 → 2026-07-05
Total claims (raw)    : 964,206
Total claims (modeled): 964,206
Pharmacies (modeled)  : 31,877
Model features        : 12
Predicted class 1     : 6,119

12 features: ['total_claims', 'total_units', 'total_benefit', 'avg_qty', 'std_qty', 'avg_cpu', 'std_cpu', 'avg_cpu_pct_wac', 'n_months

In [2]:
import nurtec_pipeline as P
import pandas as pd, numpy as np

# ---- 1. isolate ML lookalikes (high prob, tripped 0 rules) ----
look = score[score["lookalike_flag"]].copy()
print(f"Total ML lookalikes: {len(look):,}")

# ---- 2. date each lookalike by its closest-signal near-miss month ----
nm_cols = ["pharmacy_key","last_fill_month","s1_nearmiss_month",
           "s2_nearmiss_month","s4_nearmiss_month",
           "s7_best_episode_month","signal5_event_month"]
have = [c for c in nm_cols if c in score.columns]
look = look.drop(columns=[c for c in have if c != "pharmacy_key" and c in look.columns],
                 errors="ignore")
look = look.merge(score[have], on="pharmacy_key", how="left")
look = P.lookalike_recency(look, detail["ref_month"])   # -> closest_signal_event_month, months_since_closest_signal

# ---- 2b. ANCHOR to end of June 2026 (data effectively ends early July) ----
ANCHOR = pd.Timestamp("2026-06-30")
print(f"Anchor (adjusted) = {ANCHOR.date()}  [June-end, since data ends early July]")

# closest-signal event is month-resolution; compare month-start to month cutoffs
ev = pd.to_datetime(look["closest_signal_event_month"], errors="coerce")

# calendar window cutoffs (month-start comparisons)
JUN   = pd.Timestamp("2026-06-01")          # last month = June 2026
JUL24 = pd.Timestamp("2025-07-01")          # last year  = Jul 2025 .. Jun 2026 (inclusive)
# "last week of June" -> event month is June AND its day-level fill lands in Jun 24-30
last_fill_day = pd.to_datetime(look.get("last_fill_month"), errors="coerce")

# masks
mask_year  = (ev >= JUL24) & (ev <= JUN)
mask_month = (ev >= JUN)   & (ev <= JUN)     # exactly June 2026
# last week: June-month lookalikes whose most recent fill day is >= Jun 24
mask_week  = mask_month     # month-resolution fallback (see note); refined below if day data exists

windows_masks = {"last_1_year": mask_year,
                 "last_1_month": mask_month,
                 "last_1_week": mask_week}

# ---- 3. verification columns ----
ml_cols = [
    "pharmacy_key", "total_claims", "total_benefit",
    "signal_probability", "anomaly_score",
    "closest_signal", "closest_proximity", "closest_reason",
    "closest_signal_event_month", "months_since_closest_signal",
    "last_fill_month", "on_risrx",
    "p_signal1","p_signal2","p_signal3","p_signal5","p_signal7",
]
ml_cols = [c for c in ml_cols if c in look.columns]

# ---- 4. funnel ----
print("="*60)
print("ML LOOKALIKE FUNNEL — calendar-anchored to June 2026")
print("="*60)
print(f"All-time ML lookalikes: {len(look):,}\n")

with pd.ExcelWriter("Eucrisa_Funnel_ML_Lookalikes.xlsx", engine="openpyxl") as xw:
    look[ml_cols].sort_values("months_since_closest_signal").to_excel(
        xw, "all_time_lookalikes", index=False)
    for name, mask in windows_masks.items():
        sub = look[mask].copy().sort_values(
            ["months_since_closest_signal","signal_probability"], ascending=[True, False])
        print(f"{name:<14}: {len(sub):,} lookalikes | "
              f"{int(sub['total_claims'].sum()):,} claims | "
              f"${sub['total_benefit'].sum():,.0f} benefit")
        if len(sub):
            for sig, cnt in sub["closest_signal"].value_counts().items():
                print(f"      {sig:<26} {cnt:,}")
        sub[ml_cols].to_excel(xw, name, index=False)
        print()

print("Wrote Eucrisa_Funnel_ML_Lookalikes.xlsx")

KeyError: 'lookalike_flag'